# Fusion R9 reconstruction
3 GiB NF4 cache + sparse cached head + MLP padding; later varlen/direct-linear regressions disabled.

In [ ]:
import os,sys,json,subprocess,shutil,base64,bz2,hashlib
from pathlib import Path
repo=Path(os.environ["SOUP_REPO"]);results=Path(os.environ["SOUP_RESULTS"])
p=base64.b64decode('...');assert hashlib.sha256(p).hexdigest()=='6765ba57dbd8684956655628eb9e0caca5f5204f062d1e496375d9bfe833fcfa'
r=bz2.decompress(p);i=0;n=0
while i<len(r):
 j=r.index(b"\0",i);name=r[i:j].decode();i=j+1
 j=r.index(b"\0",i);size=int(r[i:j]);i=j+1
 data=r[i:i+size];i+=size
 q=repo/"solution"/name;q.parent.mkdir(parents=True,exist_ok=True);q.write_bytes(data);n+=1
print("installed",n,"R9 runtime files")


In [ ]:
run=Path("/kaggle/working/training-run")
subprocess.run([sys.executable,str(repo/"harness/run_task.py"),"--out",str(run),"--timeout","80","--only","solution"],cwd=repo,check=True)


In [ ]:
for path in run.rglob("*"):
 if path.is_file() and path.suffix in (".json",".jsonl",".log",".yaml") and path.stat().st_size<4*1024**2:
  target=results/path.relative_to(run);target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(path,target)
print((run/"result.json").read_text())
